In [1]:
x = var('x')
y = function('y')(x)

In [53]:
de = 50*diff(y,x,2) + 100*y  == 0
de

100*y(x) + 50*diff(y(x), x, x) == 0

In [55]:
f= desolve(de, y, [0,1,1]) 
f1 = diff(f, x)
f2 = diff(f, x, 2)
f



1/2*sqrt(2)*sin(sqrt(2)*x) + cos(sqrt(2)*x)

In [32]:

sum1 = 0
step = 0.1
final = 2/step 

for i in range(0,int(final)):
    ii= float(i)*step
    #y1 = pow(abs(f2(ii) + f(ii) ),2) 
    y1 = 300*pow(abs(f2(x=ii) + f(x=ii) ),2) + 30*pow(abs(1.0 - f(x=ii)),2) + 70*pow(abs(1.0-f1(x=ii)),2)
    sum1 = y1 +sum1
    
    print(ii,float(sum1),float(y1))

0.000000000000000 30.0 30.0
0.100000000000000 54.31074341909154 24.310743419091533
0.200000000000000 73.60248255367173 19.2917391345802
0.300000000000000 88.63087397460008 15.02839142092835
0.400000000000000 100.25136846262045 11.620494488020364
0.500000000000000 109.43032359907889 9.178955136458434
0.600000000000000 117.25194419755528 7.821620598476388
0.700000000000000 124.92031960147025 7.668375403914976
0.800000000000000 133.75602439286996 8.835704791399701
0.900000000000000 145.1869723634662 11.430947970596234
1.00000000000000 160.73345372251 15.5464813590438
1.10000000000000 181.98753277413616 21.25407905162614
1.20000000000000 210.58722767854337 28.599694904407222
1.30000000000000 248.1861254786906 37.59889780014724
1.40000000000000 296.4192948599761 48.2331693812855
1.50000000000000 356.8665374982455 60.447242638269415
1.60000000000000 431.01415892204056 74.14762142379507
1.70000000000000 520.2165356446767 89.20237672263615
1.80000000000000 625.6588027223343 105.44226707765755



print(float(sum1))


In [213]:
print(float(sum1))

16.83876327114647


In [185]:
pow(0,0)

1

In [186]:
log(0)

-Infinity

## Validacion de numeros duales de segundo orden (issues #18 y #25)

Estas celdas son Python compatible con SageMath: representan `[f, f', f'']` como una tupla de `float` y comparan las derivadas automaticas con diferencias finitas centrales. La bateria evita los puntos donde `log`, `sqrt` o `2/x` no son reales o finitos.

In [ ]:
from math import cos, exp, isfinite, log, sin, sqrt

# Un 2dual es equivalente a una tupla Python `(valor, primera, segunda)`.
def dual_constante(valor):
    return (float(valor), 0.0, 0.0)

def dual_variable(valor):
    return (float(valor), 1.0, 0.0)

def dual_negativo(u):
    return (-u[0], -u[1], -u[2])

def dual_suma(u, v):
    return (u[0] + v[0], u[1] + v[1], u[2] + v[2])

def dual_producto(u, v):
    return (u[0] * v[0],
            u[1] * v[0] + u[0] * v[1],
            u[2] * v[0] + 2.0 * u[1] * v[1] + u[0] * v[2])

def dual_division(u, v):
    value = u[0] / v[0]
    first = (v[0] * u[1] - u[0] * v[1]) / v[0]**2
    second = (u[2] / v[0] - 2.0 * u[1] * v[1] / v[0]**2
              + u[0] * (2.0 * v[1]**2 / v[0]**3 - v[2] / v[0]**2))
    return (value, first, second)

def dual_seno(u):
    return (sin(u[0]),
            cos(u[0]) * u[1],
            -sin(u[0]) * u[1]**2 + cos(u[0]) * u[2])

def dual_coseno(u):
    return (cos(u[0]),
            -sin(u[0]) * u[1],
            -cos(u[0]) * u[1]**2 - sin(u[0]) * u[2])

def dual_exponencial(u):
    value = exp(u[0])
    return (value, value * u[1], value * (u[1]**2 + u[2]))

def dual_logaritmo(u):
    return (log(u[0]), u[1] / u[0], u[2] / u[0] - u[1]**2 / u[0]**2)

def dual_potencia(u, v):
    # La bateria usa bases positivas, donde la regla u(x)**v(x) es real.
    value = u[0]**v[0]
    log_u = log(u[0])
    g = v[0] * u[1] / u[0] + v[1] * log_u
    derivative_of_g = (2.0 * v[1] * u[1] / u[0]
                     + v[0] * u[2] / u[0]
                     - v[0] * u[1]**2 / u[0]**2
                     + v[2] * log_u)
    return (value, value * g, value * (g**2 + derivative_of_g))

def dual_raiz(u):
    return dual_potencia(u, dual_constante(0.5))

CASES = (
    ("x", lambda x: x, lambda u: u),
    ("-x", lambda x: -x, dual_negativo),
    ("sin(x)", sin, dual_seno),
    ("cos(x)", cos, dual_coseno),
    ("exp(x)", exp, dual_exponencial),
    ("log(x+2)", lambda x: log(x + 2.0),
     lambda u: dual_logaritmo(dual_suma(u, dual_constante(2.0)))),
    ("sqrt(x+2)", lambda x: sqrt(x + 2.0),
     lambda u: dual_raiz(dual_suma(u, dual_constante(2.0)))),
    ("x+2", lambda x: x + 2.0,
     lambda u: dual_suma(u, dual_constante(2.0))),
    ("x-2", lambda x: x - 2.0,
     lambda u: dual_suma(u, dual_negativo(dual_constante(2.0)))),
    ("x*x", lambda x: x * x,
     lambda u: dual_producto(u, u)),
    ("x/2", lambda x: x / 2.0,
     lambda u: dual_division(u, dual_constante(2.0))),
    ("2/x", lambda x: 2.0 / x,
     lambda u: dual_division(dual_constante(2.0), u)),
    ("x^2", lambda x: x**2,
     lambda u: dual_potencia(u, dual_constante(2.0))),
    ("2^x", lambda x: 2.0**x,
     lambda u: dual_potencia(dual_constante(2.0), u)),
    ("x^x", lambda x: x**x,
     lambda u: dual_potencia(u, u)),
    ("sin(x^2+cos(x))",
     lambda x: sin(x**2 + cos(x)),
     lambda u: dual_seno(dual_suma(dual_potencia(u, dual_constante(2.0)),
                                  dual_coseno(u)))),
)


In [ ]:
def validar_caso(nombre, valor, dual, x, h=1e-5, tolerancia=1e-5):
    automatico = dual(dual_variable(x))
    menos = valor(x - h)
    mas = valor(x + h)
    primera = (mas - menos) / (2.0 * h)
    segunda = (mas - 2.0 * valor(x) + menos) / h**2
    errores = (abs(automatico[0] - valor(x)),
              abs(automatico[1] - primera),
              abs(automatico[2] - segunda))
    correcto = all(isfinite(error) and error <= tolerancia for error in errores)
    estado = "PASS" if correcto else "FAIL"
    print(f"{estado} {nombre} x={x:.2f} errores={errores}")
    return correcto

def validar_bateria_dual():
    puntos = (0.25, 0.5, 0.75)
    resultados = [
        validar_caso(nombre, valor, dual, x)
        for nombre, valor, dual in CASES
        for x in puntos
    ]
    aprobados = sum(resultados)
    total = len(resultados)
    estado = "PASS" if aprobados == total else "FAIL"
    print(f"{estado}: {aprobados}/{total} casos")
    if aprobados != total:
        raise AssertionError("fallo en la validacion de numeros duales")
    return aprobados, total

validar_bateria_dual()
